# Vitessce Widget Tutorial

# Example usage of Neuroglancer precomputed segmentations and annotations

This notebook demonstrates `ObsSegmentationsNgPrecomputedWrapper` and `ObsPointsNgAnnotationsWrapper`, which wrap Neuroglancer precomputed segmentation/mesh data and point-annotation data (e.g. as produced by the [tissue-map-tools](https://github.com/hms-dbmi/tissue-map-tools) library) for use with the `neuroglancer` and `layerControllerBeta` views.

The `obsSets`-type file (and its corresponding view) is required for the segmentation layer to actually resolve and display any segments -- without it, segments will not be selected/colored dynamically. A static `segments` list can alternatively be passed via `options` for quick testing without a full `obsSets` pipeline.

In [ ]:
from vitessce import (
    VitessceConfig,
    CoordinationLevel as CL,
    get_initial_coordination_scope_prefix,
    ObsSegmentationsNgPrecomputedWrapper,
    CsvWrapper,
)

## 1. Configure Vitessce

In [ ]:
vc = VitessceConfig(schema_version="1.0.17", name="Neuroglancer precomputed example")
dataset = vc.add_dataset("Melanoma")

# A Neuroglancer precomputed segmentation  meshes directory.
# fileUid here must match the value used below in link_views_by_dict's
# segmentationLayer coordination.
dataset.add_object(ObsSegmentationsNgPrecomputedWrapper(
    data_url="https://data-2.vitessce.io/data/sorger/melanoma_meshes",
    coordination_values={"fileUid": "segmentation"},
))

# An obsSets-type file is required for segments to be dynamically
# selected/colored -- obsType here must match the segmentationChannel's
# obsType coordination value set below.
dataset.add_object(CsvWrapper(
    csv_url="https://storage.googleapis.com/vitessce-demo-data/neuroglancer-march-2025/melanoma_with_embedding_filtered_ids.csv",
    data_type="obsSets",
    coordination_values={"obsType": "cell"},
    options={
        "obsIndex": "id",
        "obsSets": [{"name": "Clusters", "column": "cluster"}],
    },
))

In [ ]:
ng_view = vc.add_view("neuroglancer", dataset=dataset)
lc_view = vc.add_view("layerControllerBeta", dataset=dataset)
#  TODO: until support to load the segments is added in NG-View
# The obsSets view is not required for the segmentation to load, but
# a mounted obsSets view is needed to trigger the underlying data hook 
# that resolves obsSets data for the segmentation channel.
obs_sets_view = vc.add_view("obsSets", dataset=dataset)
vc.layout(ng_view | lc_view | obs_sets_view)

## 2. Coordinate the views

Two separate `link_views_by_dict` calls are needed:
- A plain (non-meta) link for shared spatial rendering mode and camera position/rotation.
- A multi-level (meta) link for the segmentation layer + channel, mirroring the shape
  `obsSegmentations.ng-precomputed` files require to resolve correctly.

In [ ]:
vc.link_views_by_dict([ng_view, lc_view], {
    "spatialRenderingMode": "3D",
    "spatialZoom": 0,
    "spatialTargetX": 0,
    "spatialTargetY": 0,
    "spatialTargetZ": 0,
    "spatialRotationX": 0,
    "spatialRotationY": 0,
    "spatialRotationOrbit": 0,
}, meta=False)

vc.link_views_by_dict([ng_view, lc_view], {
    "segmentationLayer": CL([{
        "fileUid": "segmentation",
        "spatialLayerOpacity": 1,
        "spatialTargetResolution": None,
        "spatialLayerVisible": True,
        "segmentationChannel": CL([{"obsType": "cell", "spatialChannelVisible": True}]),
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsSegmentations"))

## 3. Create the Vitessce widget

In [ ]:
vw = vc.widget()
vw